In [2]:
import random, sys, math

# Nota: en lugar de matrices se usan listas de listas

# Genera una matriz de distancias de nCiudades x nCiudades
def matrizDistancias(nCiud, distanciaMaxima):
    matriz = [[0 for i in range(nCiud)] for j in range(nCiud)]

    for i in range(nCiud):
        for j in range(i):
            matriz[i][j] = distanciaMaxima * random.random()
            matriz[j][i] = matriz[i][j]

    return matriz


# Elige un paso de una hormiga, teniendo en cuenta las distancias
# y las feromonas, descartando las ciudades ya visitadas.
def eligeCiudad(dists, ferom, visitadas):
    # Se calcula la tabla de pesos de cada ciudad
    listaPesos = []
    disponibles = []
    actual = visitadas[-1]

    # Influencia de cada valor (alfa: feromonas; beta: distancias)
    alfa = 1.0
    beta = 0.5

    for i in range(len(dists)):
        if i not in visitadas:
            fer = math.pow((1.0 + ferom[actual][i]), alfa)
            peso = math.pow((1.0 / dists[actual][i]), beta) * fer
            disponibles.append(i)
            listaPesos.append(peso)

    # Se elige aleatoriamente una de las ciudades disponibles,
    # teniendo en cuenta su peso relativo.
    valor = random.random() * sum(listaPesos)
    acumulado = 0.0
    i = -1

    while valor > acumulado:
        i += 1
        acumulado += listaPesos[i]

    return disponibles[i]


# Genera una "hormiga", que elegirá un camino teniendo en cuenta
# las distancias y los rastros de feromonas. Devuelve una tupla
# con el camino y su longitud.
def eligeCamino(distancias, feromonas):
    # La ciudad inicial siempre es la 0
    camino = [0]
    longCamino = 0

    # Elegir cada paso según la distancia y las feromonas
    while len(camino) < len(distancias):
        ciudad = eligeCiudad(distancias, feromonas, camino)
        longCamino += distancias[camino[-1]][ciudad]
        camino.append(ciudad)

    # Para terminar hay que volver a la ciudad de origen (0)
    longCamino += distancias[camino[-1]][0]
    camino.append(0)

    return (camino, longCamino)


# Actualiza la matriz de feromonas siguiendo el camino recibido
def rastroFeromonas(feromonas, camino, dosis):
    for i in range(len(camino) - 1):
        feromonas[camino[i]][camino[i + 1]] += dosis


# Evapora todas las feromonas multiplicándolas por una constante
# 0.9 (en otras palabras, el coeficiente de evaporación es 0.1)
def evaporaFeromonas(feromonas):
    for lista in feromonas:
        for i in range(len(lista)):
            lista[i] *= 0.9


# Resuelve el problema del viajante de comercio mediante el
# algoritmo de la colonia de hormigas.
def hormigas(distancias, iteraciones, distMedia):
    # Primero se crea una matriz de feromonas vacía
    n = len(distancias)
    feromonas = [[0 for i in range(n)] for j in range(n)]

    # El mejor camino y su longitud (inicialmente "infinita")
    mejorCamino = []
    longMejorCamino = sys.maxsize

    # En cada iteración se genera una hormiga
    for i in range(iteraciones):
        (camino, longCamino) = eligeCamino(distancias, feromonas)

        if longCamino <= longMejorCamino:
            mejorCamino = camino
            longMejorCamino = longCamino

        rastroFeromonas(feromonas, camino, distMedia / longCamino)

        # En cualquier caso, las feromonas se van evaporando
        evaporaFeromonas(feromonas)

    return (mejorCamino, longMejorCamino)


# Generación de una matriz de prueba
numCiudades = 10
distanciaMaxima = 10

ciudades = matrizDistancias(numCiudades, distanciaMaxima)

# Obtención del mejor camino
iteraciones = 1000
distMedia = numCiudades * distanciaMaxima / 2

(camino, longCamino) = hormigas(ciudades, iteraciones, distMedia)

print("Camino:", camino)
print("Longitud del camino:", longCamino)

Camino: [0, 4, 3, 2, 7, 9, 1, 5, 8, 6, 0]
Longitud del camino: 17.664662041528235


**Problema del viajero**


*Descripción general*

El Problema del Viajero (TSP, Traveling Salesman Problem) consiste en encontrar la ruta más corta posible que permita a un viajero visitar un conjunto de ciudades exactamente una vez y regresar al punto de origen. Cada ciudad está conectada con las demás mediante distancias conocidas, y el objetivo es minimizar la distancia total recorrida.

Este problema es uno de los más conocidos en el campo de la optimización combinatoria y es considerado NP-hard, lo que significa que no existe un algoritmo eficiente que garantice encontrar la solución óptima en todos los casos cuando el número de ciudades crece.

*Objetivos*
Encontrar el camino más corto posible que recorra todas las ciudades.
Minimizar el costo total del recorrido (distancia, tiempo, etc.).
Aplicar técnicas heurísticas o metaheurísticas (como colonia de hormigas) para obtener soluciones aproximadas eficientes.


*Limitaciones*

El número de posibles rutas crece factorialmente con el número de ciudades:
(n−1)!/2
Resolverlo de forma exacta es computacionalmente muy costoso para grandes cantidades de ciudades.
Las soluciones heurísticas no siempre garantizan la solución óptima, pero sí buenas aproximaciones en menor tiempo.
Trabajos previos

El problema ha sido ampliamente estudiado y abordado con diferentes enfoques:

Métodos exactos: fuerza bruta, programación dinámica (Held-Karp).
Heurísticas clásicas: vecino más cercano, inserción.
Metaheurísticas modernas:
Algoritmos genéticos
Recocido simulado
Colonia de hormigas (ACO) ← el utilizado en este código

El algoritmo de colonia de hormigas se inspira en el comportamiento real de las hormigas, que encuentran rutas óptimas mediante el uso de feromonas.